# Big Data demo

This notebook demonstrates how we can use DuckDB or Polars to work with very large datasets.

Here we will work with data from the NYC Taxi and Limousine Commission: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

This is a set of parquet files that total about 11 GB which represent individual taxi rides in new york city over the course of 10 years (about 752 million records).  

my primary interest is computing the frequencies of rides between each taxi zone, separately per month/year.  


In [1]:
import os
import time
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

from bettercode.taxi_utils import (
    download_taxi_data,
    get_data_dirs,
)

load_dotenv()
DATADIR = Path(os.getenv("DATADIR"))
orig_dir, preproc_dir = get_data_dirs(DATADIR)

In [ ]:
download_taxi_data(DATADIR, delay_between_downloads=30)

Overall progress: 100%|██████████| 120/120 [00:00<00:00, 75324.23it/s]


## Preprocess data for consistent schemas

The raw parquet files have inconsistent schemas across years (different datetime precisions, integer types, and column sets). We'll preprocess them once to create standardized files that both DuckDB and Polars can efficiently query.

In [ ]:
from bettercode.taxi_utils import preprocess_all_files

# Preprocess files to standardize schemas (only runs once unless overwrite=True)
preprocess_all_files(DATADIR, overwrite=False)

First we will load all of the individual files and determine their size and memory usage

In [ ]:
# Load all parquet files from orig_dir and add source filename column

data_files = list(orig_dir.glob("*.parquet"))
print(f"Found {len(data_files)} data files")


start_time = time.time()
# Load each file and add the source filename
memory = 0
rows = 0
for file_path in tqdm(sorted(data_files)):
    df = pd.read_parquet(file_path)
    memory += df.memory_usage(deep=True).sum()
    rows += len(df)

end_time = time.time()
print(f"\nTime to load files with pandas: {end_time - start_time:.2f} seconds")
print(f"\nTotal combined rows: {rows:,}")
print(f"Memory usage: {memory / 1024**3:.2f} GB")


Found 120 data files


100%|██████████| 120/120 [02:10<00:00,  1.09s/it]


Time to load files with pandas: 130.55 seconds

Total combined rows: 752,830,638
Memory usage: 150.56 GB


## Method 1: DuckDB Analysis

DuckDB is designed for analytical queries on large datasets and can query Parquet files directly without loading them into memory. This is extremely efficient for datasets that don't fit in RAM.

### DuckDB: Query all parquet files directly (no loading into memory)

In [16]:
# DuckDB can query parquet files directly without loading them into memory
# This is MUCH more memory efficient for large datasets

start_time = time.time()

# Query to compute zone-to-zone trip frequencies by month/year
# The preprocessed files already have PUBorough and DOBorough, so no joins needed!
query = f"""
SELECT 
    YEAR(tpep_pickup_datetime) as year,
    MONTH(tpep_pickup_datetime) as month,
    PULocationID as pickup_location_id,
    DOLocationID as dropoff_location_id,
    COUNT(*) as trip_count
FROM read_parquet('{preproc_dir}/*.parquet')
WHERE tpep_pickup_datetime IS NOT NULL
    AND YEAR(tpep_pickup_datetime) >= 2015
    AND YEAR(tpep_pickup_datetime) <= 2024
GROUP BY year, month, pickup_location_id, dropoff_location_id
ORDER BY year, month, pickup_location_id, dropoff_location_id
"""

# Execute query and get results
duckdb_results = duckdb.query(query).df()

end_time = time.time()

print(f"DuckDB analysis completed in {end_time - start_time:.2f} seconds")
print(f"Total route-month combinations: {len(duckdb_results):,}")
print(f"Memory usage of results: {duckdb_results.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Year range: {duckdb_results['year'].min()} to {duckdb_results['year'].max()}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB analysis completed in 2.06 seconds
Total route-month combinations: 3,339,788
Memory usage of results: 127.40 MB
Year range: 2015 to 2024


In [15]:
duckdb_results.head()

,year,month,pickup_location_id,dropoff_location_id,trip_count
0,2015,1,1,1,607
1,2015,1,1,48,1
2,2015,1,1,90,1
3,2015,1,1,113,1
4,2015,1,1,123,1


### DuckDB: Example queries on the results

In [6]:
# Find the most popular routes for a specific month
print("Top 10 zone-to-zone routes in January 2024:")
jan_2024 = duckdb_results[
    (duckdb_results['year'] == 2024) & (duckdb_results['month'] == 1)
].nlargest(10, 'trip_count')[['pickup_location_id', 'dropoff_location_id', 'trip_count']]
print(jan_2024)

print("\n" + "="*60 + "\n")

# Total trips by year
print("Total trips by year:")
yearly_totals = duckdb.query("""
    SELECT year, SUM(trip_count) as total_trips
    FROM duckdb_results
    GROUP BY year
    ORDER BY year
""").df()
print(yearly_totals)

Top 10 zone-to-zone routes in January 2024:
         pickup_location_id  dropoff_location_id  trip_count
2953771                 237                  236       21883
2953561                 236                  237       19402
2953560                 236                  236       15955
2953772                 237                  237       14938
2946082                 161                  237       10275
2944043                 142                  239        8980
2953709                 237                  161        8834
2946081                 161                  236        8766
2954069                 239                  142        8675
2954142                 239                  238        8445


Total trips by year:
   year  total_trips
0  2015  143505684.0
1  2016  129081079.0
2  2017  111653716.0
3  2018  101132035.0
4  2019   83635658.0
5  2020   24434353.0
6  2021   30589440.0
7  2022   39091080.0
8  2023   37852688.0
9  2024   40932294.0


### DuckDB: Create a pivot table for a specific month

In [7]:
# Create a zone-to-zone matrix for January 2024
jan_2024_data = duckdb_results[
    (duckdb_results['year'] == 2024) & (duckdb_results['month'] == 1)
]

zone_matrix = jan_2024_data.pivot(
    index='pickup_location_id',
    columns='dropoff_location_id', 
    values='trip_count'
).fillna(0).astype(int)

print(f"Zone-to-zone trip matrix for January 2024 (shape: {zone_matrix.shape}):")
print(zone_matrix.iloc[:10, :10])  # Show first 10x10 subset

Zone-to-zone trip matrix for January 2024 (shape: (259, 260)):
dropoff_location_id   1   2   3    4   5   6    7   8   9    10
pickup_location_id                                             
1                    263   0   0    0   0   1    0   0   0    0
2                      0   0   0    0   0   0    0   0   0    0
3                      0   0  12    1   0   0    1   0   0    1
4                      1   0   0  110   0   0    4   0   0    0
6                      1   0   0    0   0  12    0   0   0    0
7                      2   0   0    1   0   0  336   0   2    0
8                      0   0   0    0   0   0    0   0   0    0
9                      0   0   0    0   0   0    0   0   8    0
10                     1   0   0    2   0   0    1   0   1  106
11                     0   0   0    0   0   0    1   0   0    0


## Method 2: Polars Analysis

Polars is a DataFrame library designed for speed and efficiency. It uses lazy evaluation and can process data in parallel. Like DuckDB, it can read Parquet files efficiently.

In [8]:
# Import polars
import polars as pl

print(f"Polars version: {pl.__version__}")

Polars version: 1.37.1


### Polars: Lazy scan preprocessed parquet files

With preprocessed files that have consistent schemas, Polars can efficiently scan all files with its clean API and lazy evaluation.

In [14]:
start_time = time.time()

# For this large dataset with Polars, we need to be very careful about memory
# Strategy: Process and aggregate without joins, use streaming mode
trips_lazy = pl.scan_parquet(str(preproc_dir / "*.parquet"))

# Streaming aggregation: filter, extract date parts, group by, aggregate
# This should use minimal memory by processing in chunks
polars_results = (
    trips_lazy
    .select([
        pl.col("tpep_pickup_datetime"),
        pl.col("PULocationID"),
        pl.col("DOLocationID")
    ])
    .filter(pl.col("tpep_pickup_datetime").is_not_null())
    .with_columns([
        pl.col("tpep_pickup_datetime").dt.year().alias("year"),
        pl.col("tpep_pickup_datetime").dt.month().alias("month"),
    ])
    .filter(
        (pl.col("year") >= 2015) & 
        (pl.col("year") <= 2024) 
    )
    .select(["year", "month", "PULocationID", "DOLocationID"])
    .group_by(["year", "month", "PULocationID", "DOLocationID"])
    .len()
    .sort(["year", "month", "PULocationID", "DOLocationID"])
    .collect(engine="streaming")  # Use streaming engine for memory efficiency
)

# Rename and add zone names to the aggregated result
polars_results = (
    polars_results
    .rename({"len": "trip_count"})
)

end_time = time.time()

print(f"Polars analysis completed in {end_time - start_time:.2f} seconds")
print(f"Total route-month combinations: {len(polars_results):,}")
print(f"Memory usage of results: {polars_results.estimated_size() / 1024**2:.2f} MB")
print(f"Year range: {polars_results['year'].min()} to {polars_results['year'].max()}")
polars_results.head(10)

Polars analysis completed in 3.42 seconds
Total route-month combinations: 3,339,788
Memory usage of results: 79.63 MB
Year range: 2015 to 2024


year,month,PULocationID,DOLocationID,trip_count
i32,i8,i64,i64,u32
2015,1,1,1,607
2015,1,1,48,1
2015,1,1,90,1
2015,1,1,113,1
2015,1,1,123,1
2015,1,1,132,2
2015,1,1,138,1
2015,1,1,145,1
2015,1,1,161,1


### Polars: Example queries on the results

In [10]:
# Find the most popular routes for January 2024
print("Top 10 zone-to-zone routes in January 2024 (Polars):")
jan_2024_polars = (
    polars_results
    .filter((pl.col("year") == 2024) & (pl.col("month") == 1))
    .sort("trip_count", descending=True)
    .select(["PULocationID",  "DOLocationID",  "trip_count"])
    .head(10)
)
print(jan_2024_polars)

print("\n" + "="*60 + "\n")

# Total trips by year
print("Total trips by year (Polars):")
yearly_totals_polars = (
    polars_results
    .group_by("year")
    .agg(pl.col("trip_count").sum().alias("total_trips"))
    .sort("year")
)
print(yearly_totals_polars)

Top 10 zone-to-zone routes in January 2024 (Polars):
shape: (10, 3)
┌──────────────┬──────────────┬────────────┐
│ PULocationID ┆ DOLocationID ┆ trip_count │
│ ---          ┆ ---          ┆ ---        │
│ i64          ┆ i64          ┆ u32        │
╞══════════════╪══════════════╪════════════╡
│ 237          ┆ 236          ┆ 21883      │
│ 236          ┆ 237          ┆ 19402      │
│ 236          ┆ 236          ┆ 15955      │
│ 237          ┆ 237          ┆ 14938      │
│ 161          ┆ 237          ┆ 10275      │
│ 142          ┆ 239          ┆ 8980       │
│ 237          ┆ 161          ┆ 8834       │
│ 161          ┆ 236          ┆ 8766       │
│ 239          ┆ 142          ┆ 8675       │
│ 239          ┆ 238          ┆ 8445       │
└──────────────┴──────────────┴────────────┘


Total trips by year (Polars):
shape: (10, 2)
┌──────┬─────────────┐
│ year ┆ total_trips │
│ ---  ┆ ---         │
│ i32  ┆ u32         │
╞══════╪═════════════╡
│ 2015 ┆ 143505684   │
│ 2016 ┆ 129081079   │
│ 20

### Verify DuckDB and Polars Results Match

Let's compare the results from both methods to ensure they're computing the same values.

In [11]:
# Convert Polars results to Pandas for comparison
polars_df = polars_results.to_pandas()

# Sort both dataframes the same way for comparison
duckdb_sorted = duckdb_results.sort_values(['year', 'month', 'pickup_location_id', 'dropoff_location_id']).reset_index(drop=True)
polars_sorted = polars_df.sort_values(['year', 'month', 'PULocationID', 'DOLocationID']).reset_index(drop=True)

# Rename Polars columns to match DuckDB
polars_sorted = polars_sorted.rename(columns={
    'PULocationID': 'pickup_location_id',
    'DOLocationID': 'dropoff_location_id'
})

# Compare the key columns (year, month, location IDs, trip_count)
cols_to_compare = ['year', 'month', 'pickup_location_id', 'dropoff_location_id', 'trip_count']

print("Comparing DuckDB and Polars results:")
print(f"DuckDB rows: {len(duckdb_sorted):,}")
print(f"Polars rows: {len(polars_sorted):,}")
print(f"Rows match: {len(duckdb_sorted) == len(polars_sorted)}")

# Check if values match
if len(duckdb_sorted) == len(polars_sorted):
    comparison = duckdb_sorted[cols_to_compare].equals(polars_sorted[cols_to_compare])
    print(f"\nAll values match: {comparison}")
    
    if not comparison:
        # Find differences
        mask = (duckdb_sorted[cols_to_compare] != polars_sorted[cols_to_compare]).any(axis=1)
        differences = duckdb_sorted[mask][cols_to_compare]
        print(f"\nFound {len(differences)} differing rows:")
        print(differences.head(10))
    else:
        print("\n✓ Results are identical!")
        
    # Compare totals
    duckdb_total = duckdb_sorted['trip_count'].sum()
    polars_total = polars_sorted['trip_count'].sum()
    print(f"\nTotal trips (DuckDB): {duckdb_total:,}")
    print(f"Total trips (Polars): {polars_total:,}")
    print(f"Difference: {abs(duckdb_total - polars_total):,}")
else:
    print("\n⚠ Different number of rows - investigating...")
    # Show which combinations are in one but not the other
    duckdb_keys = set(zip(duckdb_sorted['year'], duckdb_sorted['month'], 
                          duckdb_sorted['pickup_location_id'], duckdb_sorted['dropoff_location_id']))
    polars_keys = set(zip(polars_sorted['year'], polars_sorted['month'],
                          polars_sorted['pickup_location_id'], polars_sorted['dropoff_location_id']))
    
    only_duckdb = duckdb_keys - polars_keys
    only_polars = polars_keys - duckdb_keys
    
    print(f"Combinations only in DuckDB: {len(only_duckdb)}")
    print(f"Combinations only in Polars: {len(only_polars)}")

Comparing DuckDB and Polars results:
DuckDB rows: 3,292,994
Polars rows: 3,292,994
Rows match: True

All values match: False

Found 0 differing rows:
Empty DataFrame
Columns: [year, month, pickup_location_id, dropoff_location_id, trip_count]
Index: []

Total trips (DuckDB): 741,908,027
Total trips (Polars): 741,908,027
Difference: 0.0


### Polars: Create a pivot table for a specific month

In [12]:
# Create a zone-to-zone matrix for January 2024 using Polars pivot
jan_2024_polars_data = polars_results.filter(
    (pl.col("year") == 2024) & (pl.col("month") == 1)
)

# Polars native pivot
zone_matrix_polars = jan_2024_polars_data.pivot(
    values="trip_count",
    index="PULocationID",
    columns="DOLocationID",
    aggregate_function="sum"
).fill_null(0)

print(f"Zone-to-zone trip matrix for January 2024 (Polars) - shape: {zone_matrix_polars.shape}:")
print(zone_matrix_polars.head(10).select(zone_matrix_polars.columns[:11]))  # Show first 10 rows and 11 columns

Zone-to-zone trip matrix for January 2024 (Polars) - shape: (259, 261):
shape: (10, 11)
┌──────────────┬─────┬─────┬─────┬───┬─────┬─────┬─────┬─────┐
│ PULocationID ┆ 1   ┆ 6   ┆ 48  ┆ … ┆ 230 ┆ 265 ┆ 70  ┆ 216 │
│ ---          ┆ --- ┆ --- ┆ --- ┆   ┆ --- ┆ --- ┆ --- ┆ --- │
│ i64          ┆ u32 ┆ u32 ┆ u32 ┆   ┆ u32 ┆ u32 ┆ u32 ┆ u32 │
╞══════════════╪═════╪═════╪═════╪═══╪═════╪═════╪═════╪═════╡
│ 1            ┆ 263 ┆ 1   ┆ 1   ┆ … ┆ 1   ┆ 13  ┆ 0   ┆ 0   │
│ 2            ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ 0   ┆ 0   ┆ 1   ┆ 1   │
│ 3            ┆ 0   ┆ 0   ┆ 1   ┆ … ┆ 0   ┆ 2   ┆ 0   ┆ 0   │
│ 4            ┆ 1   ┆ 0   ┆ 61  ┆ … ┆ 23  ┆ 4   ┆ 0   ┆ 2   │
│ 6            ┆ 1   ┆ 12  ┆ 0   ┆ … ┆ 0   ┆ 2   ┆ 0   ┆ 0   │
│ 7            ┆ 2   ┆ 0   ┆ 7   ┆ … ┆ 25  ┆ 2   ┆ 6   ┆ 6   │
│ 8            ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ 0   ┆ 3   ┆ 0   ┆ 0   │
│ 9            ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ 0   ┆ 0   ┆ 0   ┆ 2   │
│ 10           ┆ 1   ┆ 0   ┆ 28  ┆ … ┆ 34  ┆ 21  ┆ 0   ┆ 13  │
│ 11           ┆ 0   ┆ 0   ┆ 0

/var/folders/r2/f85nyfr1785fj4257wkdj7480000gn/T/ipykernel_80337/2141646222.py:7: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  zone_matrix_polars = jan_2024_polars_data.pivot(


## Comparison: DuckDB vs Polars vs Pandas

Let's compare the three approaches for working with large datasets:

## Method 3: Dask Analysis

Dask is a parallel computing library that provides familiar pandas-like APIs with lazy evaluation. It can handle datasets larger than memory by processing data in chunks and can scale from a single machine to a cluster.

In [5]:
# Import dask
import dask
import dask.dataframe as dd

print(f"Dask version: {dask.__version__}")

Dask version: 2026.1.1


### Dask: Lazy load preprocessed parquet files

Dask uses lazy evaluation similar to Polars. Operations build up a task graph that is only executed when you call `.compute()`. This allows Dask to optimize the computation and process data in chunks.

In [6]:
start_time = time.time()

# Read all parquet files lazily
trips_dask = dd.read_parquet(preproc_dir / "*.parquet")

# Filter, extract date parts, and aggregate
# Dask uses pandas-like API with lazy evaluation
dask_results = (
    trips_dask
    .dropna(subset=["tpep_pickup_datetime"])
    .assign(
        year=lambda df: df["tpep_pickup_datetime"].dt.year,
        month=lambda df: df["tpep_pickup_datetime"].dt.month,
    )
    .query("year >= 2015 and year <= 2024")
    .groupby(["year", "month", "PULocationID", "DOLocationID"])
    .size()
    .reset_index()
    .rename(columns={0: "trip_count"})
    .compute()  # Execute the computation
)

# Sort results
dask_results = dask_results.sort_values(
    ["year", "month", "PULocationID", "DOLocationID"]
).reset_index(drop=True)

end_time = time.time()

print(f"Dask analysis completed in {end_time - start_time:.2f} seconds")
print(f"Total route-month combinations: {len(dask_results):,}")
print(f"Memory usage of results: {dask_results.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Year range: {dask_results['year'].min()} to {dask_results['year'].max()}")
dask_results.head(10)

Dask analysis completed in 12.87 seconds
Total route-month combinations: 3,339,788
Memory usage of results: 101.92 MB
Year range: 2015 to 2024


,year,month,PULocationID,DOLocationID,trip_count
0,2015,1,1,1,607
1,2015,1,1,48,1
2,2015,1,1,90,1
3,2015,1,1,113,1
4,2015,1,1,123,1
5,2015,1,1,132,2
6,2015,1,1,138,1
7,2015,1,1,145,1
8,2015,1,1,161,1
9,2015,1,1,162,1


### Dask: Example queries on the results

In [ ]:
# Find the most popular routes for January 2024
print("Top 10 zone-to-zone routes in January 2024 (Dask):")
jan_2024_dask = (
    dask_results
    .query("year == 2024 and month == 1")
    .nlargest(10, "trip_count")[["PULocationID", "DOLocationID", "trip_count"]]
)
print(jan_2024_dask)

print("\n" + "="*60 + "\n")

# Total trips by year
print("Total trips by year (Dask):")
yearly_totals_dask = (
    dask_results
    .groupby("year")["trip_count"]
    .sum()
    .reset_index()
    .rename(columns={"trip_count": "total_trips"})
    .sort_values("year")
)
print(yearly_totals_dask)

### Dask: Create a pivot table for a specific month

In [ ]:
# Create a zone-to-zone matrix for January 2024 using Dask results
# Note: dask_results is already a pandas DataFrame after .compute()
jan_2024_dask_data = dask_results.query("year == 2024 and month == 1")

zone_matrix_dask = jan_2024_dask_data.pivot(
    index="PULocationID",
    columns="DOLocationID",
    values="trip_count"
).fillna(0).astype(int)

print(f"Zone-to-zone trip matrix for January 2024 (Dask) - shape: {zone_matrix_dask.shape}:")
print(zone_matrix_dask.iloc[:10, :10])  # Show first 10x10 subset

### Key Differences:

**DuckDB:**
- SQL-based interface (familiar for SQL users)
- Queries parquet files directly without loading into memory
- Excellent for analytical queries on data that doesn't fit in RAM
- Can create persistent databases for repeated queries
- Best for: SQL users, datasets larger than RAM, persistent storage

**Polars:**
- DataFrame API (similar to Pandas but faster)
- Uses lazy evaluation for query optimization
- Parallel processing by default
- Memory-efficient with columnar storage
- Best for: DataFrame users, data pipeline processing, speed-critical applications

**Dask:**
- Pandas-like API (easiest transition from pandas)
- Uses lazy evaluation with task graphs
- Scales from single machine to distributed clusters
- Integrates well with the Python ecosystem (NumPy, scikit-learn)
- Best for: Pandas users, scaling existing code, distributed computing

**Pandas (from earlier cell):**
- Traditional DataFrame API (most familiar)
- Loads entire dataset into memory
- Single-threaded processing
- Best for: Smaller datasets that fit in RAM, quick prototyping